# Flujo de información para generar capa regional 
#### Modelo de estimación de demanda eléctrica MERLIN_EDM
--- 
En este Notebook se empleará el modelo ``merlin_edm`` entrenado pra obtener la construcción de capas geoespaciales de demanda eléctrica en escala regional y desagregada por sector de interés. 

Los archivos de entrada para esto serán: 

- ``../data/raw/wp2_elec_input_sector_shares_raw.csv``: Los shares regionales provenientes del Balance Regional de Energía que están en formato "largo", es decir, el archivo viene como ``año | región | sector | valor`` y con datos disponibles hasta el año 2024.
- ``../data/rec_2024_2025/temperatura_regional_2024_2025.parquet``: Es la temperatura de los años 2024 y 2025 en todo Chile, en escala regional y resolución horaria. 
- ``../data/raw/reg_alias.json``: Son los alias de formato ISO de la región con respecto a su nombre disponible en las bases geoespaciales. Sirve para cruzar la información del balance regional de energía con el de las temperaturas. 

El procesamiento de estos archivos de entrada llevarán a lo siguiente: 

- Generación de rezagos temporales de temperatura (temperatura presente y 7 lags hacia atrás).
- Cálculo de series trigonométricas de hora/semana/año. 
- Condicionales de día hábil/fin de semana/feriado en escala regional. 
- Intensidades energéticas ``total_region/total_nacional`` y por sector ``total_sector_region/total_regional``.

Se debería generar una matriz de inputs que contenga los datos de todo el país para poder tomarlo como inferencia del modelo. El orden de los inputs importa para la red neuronal. Este se guarda en un archivo de texto llamado ``../data/rec_2024_2025/columns.txt`` y para cargarlo a la sesión el código es: 

```python
columns = []  # Lista vacía para que se guarden los nombres de las columnas
with open("../data/rec_2024_2025/columns.txt", "r") as f:
    for line in f: 
        columns.append(line.strip()) 

```

La matriz de inputs se usa para el modelo de red neuronal entrenado. Los outputs que se deberían obtener son: 

- ``../data/rec_2024_2025/results/capas_regionales/wp2_output_demanda_electrica_regional.gpkg``: Geocapa con los totales anuales (año 2024 y 2025) en cada región, total y por sector (RCPIT)
- ``../data/rec_2024_2025/results/capas_regionales/wp2_output_demanda_electrica_regional_ts.parquet``: Serie de tiempo con la demanda de electricidad (año 2024 y 2025) en cada región, total y por sector (RCPIT)

In [7]:
# ==========================================
# CELDA 1: CONFIGURACIÓN E IMPORTACIONES
# ==========================================
import os
import sys
import pandas as pd
import joblib
from tensorflow.keras.models import load_model
import numpy as np
import matplotlib.pyplot as plt

# 1. Configuración de Rutas Globales
BASE_DIR = os.path.abspath("..")
MODEL_PATH = os.path.join(BASE_DIR, "models", "ds_comunal", "best_merlin_mlp_global.keras")
TEMP_SCALER_PATH = os.path.join(BASE_DIR, "models", "scaler_temp_global.pkl")
HIST_SHARES_PATH = os.path.join(BASE_DIR, "data", "raw", "wp2_elec_input_sector_shares_raw.csv")
TEMP_REG_PATH = os.path.join(BASE_DIR, "data", "rec_2024_2025", "temperatura_regional_2024_2025.parquet")
COLUMNS_PATH = os.path.join(BASE_DIR, "data", "rec_2024_2025", "columns.txt")
OUT_DIR = os.path.join(BASE_DIR, "data", "rec_2024_2025", "results", "capas_regionales")

os.makedirs(OUT_DIR, exist_ok=True)

# 2. Parámetros del Modelo
IS_COMUNA = 0  # Trabajamos con regiones ahora
SECTORES = ['I', 'R', 'C', 'P', 'T']
A_PARAM = np.exp(-1.1315)  
B_PARAM = 0.8988
AÑOS_TARGET = [2024, 2025]

# 3. Cargar columnas
columns = []  # Lista vacía para que se guarden los nombres de las columnas
with open("../data/rec_2024_2025/columns.txt", "r") as f:
    for line in f: 
        columns.append(line.strip()) 

# 4. Cargar modelos de red neuronal y scaler de temperatura
model = load_model(MODEL_PATH)
temp_scaler = joblib.load(TEMP_SCALER_PATH)

# 5. Cargar los shares regionales
df_hist_shares = pd.read_csv(HIST_SHARES_PATH)

# 6. Cargar los shares de temperatura
df_temp_global = pd.read_parquet(TEMP_REG_PATH)
regiones = df_temp_global["region"].unique().tolist()

In [ ]:
# ==========================================
# CELDA 2: EXTRAPOLACIÓN DE SHARES Y RATIOS
# ==========================================

print("Iniciando extrapolación de consumos y cálculo de shares...")

# 1. Modificar la arquitectura del df de shares regionales
# (Nota: Verifica si tu columna se llama 'región' o 'region', aquí usaré 'region' sin tilde)
df_sec = df_hist_shares.pivot_table(
        index=['año', 'región'], 
        columns='sector', 
        values='valor', 
        aggfunc='sum'
    ).reset_index()

# Llenar posibles nulos con 0 (por si alguna región no tiene un sector específico)
df_sec = df_sec.fillna(0)

sectores = ['Industrial', 'Residencial', 'Comercial', 'Público', 'Transporte']
sectores_alias = {'Industrial': "I", 'Residencial': "R", 'Comercial': "C", 'Público': "P", 'Transporte': "T"}

# Verificación de seguridad: si un sector no existe en la tabla, lo creamos con 0
for sec in sectores:
    if sec not in df_sec.columns:
        df_sec[sec] = 0.0

# 2. Iterar por región para aislar la historia y hacer la regresión
proyecciones = []
regiones = df_sec["región"].unique().tolist()

for region in regiones:
    # Aislamos la historia exclusiva de esta región y la ordenamos cronológicamente
    df_region = df_sec[df_sec['región'] == region].sort_values('año').copy()
    proyecciones.append(df_region)
    
    years_hist = df_region['año'].values
    
    # 3. Proyectar para los años objetivo (2024, 2025)
    for target_year in AÑOS_TARGET:
        # Si el año ya viene en los datos originales, no lo sobreescribimos
        if target_year in years_hist:
            continue
            
        nueva_fila = {'año': target_year, 'región': region}
        
        for sec in sectores:
            if len(years_hist) < 2:
                # No hay historia suficiente para trazar una recta, copiamos el último año
                nueva_fila[sec] = df_region.iloc[-1][sec]
            else:
                # Extrapolación lineal exclusiva para este sector y región
                z = np.polyfit(years_hist, df_region[sec].values, 1)
                p = np.poly1d(z)
                # FILTRO FÍSICO: El consumo proyectado nunca puede ser negativo
                nueva_fila[sec] = max(0.0, p(target_year))
        
        proyecciones.append(pd.DataFrame([nueva_fila]))

# 4. Unir todo el historial + el futuro en un solo DataFrame
df_shares_proyectados = pd.concat(proyecciones, ignore_index=True)

# 5. Calcular los totales y los Shares definitivos
df_shares_proyectados['total_consumo_region'] = df_shares_proyectados[sectores].sum(axis=1)

for sec in sectores:
    alias = sectores_alias[sec] # Extrae I, R, C, P o T
    nombre_columna = f'share_{alias}'
    
    # Fracción = Sector / Total (usamos np.where para evitar división por cero)
    df_shares_proyectados[nombre_columna] = np.where(
        df_shares_proyectados['total_consumo_region'] > 0,
        df_shares_proyectados[sec] / df_shares_proyectados['total_consumo_region'],
        0.0
    )

# 6. Calcular el consumo total nacional por año
df_nacional = df_shares_proyectados.groupby('año')['total_consumo_region'].sum().reset_index()
df_nacional.rename(columns={'total_consumo_region': 'total_consumo_nacional'}, inplace=True)

# Unir el total nacional de vuelta al dataframe original
df_shares_proyectados = pd.merge(df_shares_proyectados, df_nacional, on='año', how='left')

# 7. Calcular la magnitud de la región (region_share)
df_shares_proyectados['region_comuna_share'] = df_shares_proyectados['total_consumo_region'] / df_shares_proyectados['total_consumo_nacional']

print("¡Proyección de shares completada!")

Iniciando extrapolación de consumos y cálculo de shares...
¡Proyección de shares completada!


In [ ]:
# Calcular lags de temperatura